# ⏱️ End-to-End Pipeline Latency — with a real Azure endpoint

**Birlasoft FORGE FDE Academy · Sprint 1 Day 16 · DecisionStream AI**

---

Your session outcome says: **"achieving P95 < 5 seconds for a single document."**

By the end of this notebook you will have measured your own endpoint, found
that the ingestion path is nowhere near five seconds, and understood why that
is not a failure to optimise.

### What is real here and what is modelled

| Part | Real or modelled |
|---|---|
| Generation latency, P50 / P95 / P99 | **Real** — measured against your endpoint |
| The long tail | **Real** — and it is the point |
| Time to first token, streaming | **Real** — measured |
| What output length costs you | **Real** — measured |
| What chain of thought costs | **Real** — measured |
| Document Intelligence and retrieval timings | **Modelled** — unless you supply those endpoints too |
| The composed pipeline | **Arithmetic** over the above |

Everything that carries the lesson is measured. The modelled parts are clearly
labelled wherever they appear.

### Why a real endpoint matters for this lesson specifically

Latency is not a bell curve. It is mostly fast with an occasional long tail —
a cold start, a retry, an unusually long completion, a noisy neighbour.

If you model it as a normal distribution you will conclude that P95 sits close
to the mean, spend your optimisation effort on the mean, and change nothing.

**You cannot be convinced of that by a simulation.** You have to watch your own
endpoint produce a 9-second response in a run whose median was 1.8 seconds.

---

### Before you run anything

**Your key is a credential.** The notebook asks with a masked prompt and never
prints it. A shared Colab notebook carries its outputs — do not paste the key
into a cell "just for a second", because it is then in the file and in your
revision history.

**This costs money.** Every call is billed. The defaults are deliberately
small — around **60 short calls** for the whole notebook. Raise `N_SAMPLES`
once you have seen it work, but know that a good tail estimate wants a few
hundred samples and that is a real spend.

## 1 · Install and import

In [ ]:
%pip install -q openai numpy matplotlib

import os, re, json, time, math, statistics, getpass, warnings, concurrent.futures
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.figsize"] = (10, 3.4)
matplotlib.rcParams["axes.grid"] = True
matplotlib.rcParams["grid.alpha"] = 0.25
warnings.filterwarnings("ignore")
print("imports ready")

## 2 · Your Azure credentials

Paste when prompted. The key is masked and never stored in the notebook.

**What you need:** an Azure OpenAI or AI Foundry resource with a deployed chat
model. Portal → your resource → *Keys and Endpoint*, plus the **deployment
name** you gave the model (not the model name — the deployment name).

Leave the endpoint blank to run in **simulated mode**: every cell works, the
numbers come from a lognormal-with-tail model instead of your endpoint, and
nothing is billed. The lessons are identical; the conviction is weaker.

In [ ]:
AOAI_ENDPOINT   = os.environ.get("AOAI_ENDPOINT", "").strip()
AOAI_KEY        = os.environ.get("AOAI_KEY", "").strip()
AOAI_DEPLOYMENT = os.environ.get("AOAI_DEPLOYMENT", "").strip()
API_VERSION     = "2024-10-21"   # VERIFY against current docs before you rely on it

if not AOAI_ENDPOINT:
    AOAI_ENDPOINT = input("Azure OpenAI / Foundry endpoint (blank = simulated): ").strip()
if AOAI_ENDPOINT:
    if not AOAI_KEY:
        AOAI_KEY = getpass.getpass("API key (hidden): ").strip()
    if not AOAI_DEPLOYMENT:
        AOAI_DEPLOYMENT = input("Deployment name (e.g. gpt-4o-mini): ").strip()

LIVE = bool(AOAI_ENDPOINT and AOAI_KEY and AOAI_DEPLOYMENT)

# How many calls per experiment. Every one is billed.
N_SAMPLES = 30          # the main latency distribution
N_STREAM  = 8           # streaming / time-to-first-token
N_LENGTH  = 5           # per output-length bucket

client = None
if LIVE:
    from openai import AzureOpenAI
    client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, api_key=AOAI_KEY,
                         api_version=API_VERSION, max_retries=0, timeout=90.0)
    print(f"mode:       LIVE — real measurements")
    print(f"endpoint:   {AOAI_ENDPOINT[:38]}...")
    print(f"deployment: {AOAI_DEPLOYMENT}")
    print(f"budget:     ~{N_SAMPLES + N_STREAM + N_LENGTH*3 + 8} calls for the whole notebook")
    print("\nNOTE: max_retries=0 deliberately. A retry hides a tail event, and the")
    print("tail is exactly what we came to measure.")
else:
    print("mode: SIMULATED — nothing billed. Re-run with credentials for the real thing.")

## 3 · One call, and what it is made of

Before measuring a distribution, look at a single request. This is the habit:
**understand the shape of one thing before you average a thousand.**

In [ ]:
SYSTEM = ("You assess motor insurance claims. Answer only from the case text. "
          "Every figure carries its source. Never state a decision.")

CASE = """
Claim CLM-2026-4471. Front nearside collision, 14 March 2026 (page 1).
Repair estimate total GBP 12,400.00 (page 7). Policy limit GBP 25,000.00,
excess GBP 350.00 (page 3). Supplementary estimate GBP 14,750.00 (page 11).
Vehicle pre-accident market value GBP 8,400.00 (page 1).
"""

TASK = "Summarise the claim in three sentences for a handler. Cite page numbers."


def simulate_ms(median=1800, sigma=0.45, tail_p=0.07, tail_mult=4.0, rng=None):
    """Lognormal body plus an explicit tail event. NOT a bell curve."""
    rng = rng or np.random.default_rng()
    v = math.exp(math.log(median) + rng.normal(0, sigma))
    if rng.random() < tail_p:
        v *= tail_mult
    return v


def one_call(max_tokens=220, task=TASK, temperature=0.0):
    """Returns (elapsed_ms, text, usage_dict). Never raises — errors are data."""
    t0 = time.perf_counter()
    if not LIVE:
        # The simulation has to respond to output length, or cells 7 and 8
        # teach the opposite of the truth. Roughly: a fixed overhead plus a
        # per-token cost, then the same lognormal spread and tail.
        want = re.search(r"about (\d+) words", task)
        asked = int(want.group(1)) * 1.4 if want else 110      # words -> tokens
        out_tok = max(20, min(max_tokens * 0.85, asked))
        med = 450 + out_tok * 9.5
        ms = simulate_ms(median=med)
        time.sleep(min(ms / 8000, 0.20))
        return ms, "(simulated completion)", {
            "prompt_tokens": 190, "completion_tokens": int(out_tok), "simulated": True}
    try:
        r = client.chat.completions.create(
            model=AOAI_DEPLOYMENT,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"{CASE}\n\nTASK: {task}"}],
            max_tokens=max_tokens, temperature=temperature)
        ms = (time.perf_counter() - t0) * 1000
        u = r.usage
        return ms, (r.choices[0].message.content or ""), {
            "prompt_tokens": getattr(u, "prompt_tokens", 0),
            "completion_tokens": getattr(u, "completion_tokens", 0),
            "model": getattr(r, "model", AOAI_DEPLOYMENT)}
    except Exception as e:
        ms = (time.perf_counter() - t0) * 1000
        return ms, "", {"error": f"{type(e).__name__}: {str(e)[:120]}"}


ms, text, usage = one_call()
print(f"elapsed: {ms:,.0f} ms")
print(f"usage:   {usage}")
print(f"\n{text[:400]}")

if "error" in usage:
    print("\nThat call failed. Check the deployment name and endpoint. The notebook")
    print("will keep running — failures are recorded rather than hidden.")

## 4 · Measure the distribution

This is the cell that costs money and the cell that teaches the lesson.

`N_SAMPLES` sequential calls, identical prompt, identical settings. Any spread
you see is the service, not your request.

Watch the running maximum. That is the number your P95 is made of.

In [ ]:
samples, errors = [], 0
print(f"running {N_SAMPLES} calls ({'live' if LIVE else 'simulated'})\n")
print(f"{'#':>4} {'ms':>9}  {'out tok':>8}   running p50 / p95 / max")
print("-" * 66)

for i in range(N_SAMPLES):
    ms, _, u = one_call()
    if "error" in u:
        errors += 1
        print(f"{i+1:>4} {ms:>9,.0f}  {'ERROR':>8}   {u['error'][:34]}")
        continue
    samples.append(ms)
    a = np.array(samples)
    print(f"{i+1:>4} {ms:>9,.0f}  {u.get('completion_tokens', 0):>8}   "
          f"{np.percentile(a,50):>7,.0f} / {np.percentile(a,95):>7,.0f} / {a.max():>7,.0f}")

gen = np.array(samples)
GEN = {"p50": float(np.percentile(gen,50)), "p95": float(np.percentile(gen,95)),
       "p99": float(np.percentile(gen,99)), "mean": float(gen.mean()),
       "min": float(gen.min()), "max": float(gen.max()), "n": len(gen),
       "errors": errors}

print("\n" + "=" * 66)
for k in ("n","errors","min","mean","p50","p95","p99","max"):
    v = GEN[k]
    print(f"  {k:<8} {v:>10,.0f}" if isinstance(v,float) else f"  {k:<8} {v:>10}")

## 5 · The shape of it

Two things to look for.

**The mean sits left of centre.** A handful of slow calls drag it right of the
median but nowhere near P95 — which is why a mean-based alert on this pipeline
will sit quietly through a genuine degradation.

**The gap between P50 and P95** is what your users experience as "sometimes it
is slow". That gap is not noise to be averaged away. It is the product.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6))

ax1.hist(gen, bins=min(18, max(6, len(gen)//2)), color="#8955FD",
         edgecolor="white", alpha=0.85)
for lbl, val, col in (("mean", GEN["mean"], "#857DA0"),
                      ("P50",  GEN["p50"],  "#2E0C69"),
                      ("P95",  GEN["p95"],  "#FD600E"),
                      ("max",  GEN["max"],  "#C0223B")):
    ax1.axvline(val, color=col, linestyle="--", linewidth=1.6)
    ax1.text(val, ax1.get_ylim()[1]*0.92, f" {lbl}", color=col, fontsize=8.5,
             rotation=90, va="top")
ax1.set_title("Generation latency — where the calls landed", fontsize=11)
ax1.set_xlabel("ms")

s = np.sort(gen)
ax2.plot(np.arange(1, len(s)+1)/len(s)*100, s, marker="o", ms=3,
         color="#2E0C69", linewidth=1.4)
ax2.axhline(GEN["p95"], color="#FD600E", linestyle="--", linewidth=1.4)
ax2.text(2, GEN["p95"], " P95", color="#FD600E", fontsize=9, va="bottom")
ax2.set_title("Sorted — the tail is the right-hand end", fontsize=11)
ax2.set_xlabel("percentile"); ax2.set_ylabel("ms")
plt.tight_layout(); plt.show()

spread = GEN["p95"] / GEN["p50"]
print(f"P95 is {spread:.1f}x the median.")
print(f"The slowest call was {GEN['max']/GEN['p50']:.1f}x the median.")
print(f"The mean sits at the {float((gen < GEN['mean']).mean())*100:.0f}th percentile "
      f"— which is why it tells you almost nothing about a bad day.")

## 6 · Streaming — what it buys, measured

Streaming does not make anything faster. It changes **when the reader sees the
first thing**, which is what they experience as responsiveness.

This cell measures both numbers on the same calls: time to first token, and
time to completion.

In [ ]:
def one_stream(max_tokens=220):
    """Returns (ttft_ms, total_ms, n_chunks)."""
    t0 = time.perf_counter()
    if not LIVE:
        total = simulate_ms()
        n = 24
        return total / n, total, n
    try:
        stream = client.chat.completions.create(
            model=AOAI_DEPLOYMENT,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"{CASE}\n\nTASK: {TASK}"}],
            max_tokens=max_tokens, temperature=0.0, stream=True)
        ttft, n = None, 0
        for chunk in stream:
            if not chunk.choices:
                continue
            d = chunk.choices[0].delta
            if getattr(d, "content", None):
                n += 1
                if ttft is None:
                    ttft = (time.perf_counter() - t0) * 1000
        total = (time.perf_counter() - t0) * 1000
        return (ttft if ttft is not None else total), total, n
    except Exception as e:
        t = (time.perf_counter() - t0) * 1000
        print(f"  stream failed: {type(e).__name__}: {str(e)[:80]}")
        return t, t, 0

ttfts, totals = [], []
print(f"{'#':>3} {'first token':>13} {'total':>10} {'chunks':>8}")
print("-" * 40)
for i in range(N_STREAM):
    a, b, n = one_stream()
    ttfts.append(a); totals.append(b)
    print(f"{i+1:>3} {a:>11,.0f}ms {b:>8,.0f}ms {n:>8}")

STREAM = {"ttft_p50": float(np.percentile(ttfts,50)),
          "ttft_p95": float(np.percentile(ttfts,95)),
          "total_p50": float(np.percentile(totals,50)),
          "total_p95": float(np.percentile(totals,95))}

print(f"""
{'':<34}{'P50':>10}{'P95':>10}
{'-'*54}
{'total generation time':<34}{STREAM['total_p50']:>9,.0f}ms{STREAM['total_p95']:>9,.0f}ms
{'first visible text — no streaming':<34}{STREAM['total_p50']:>9,.0f}ms{STREAM['total_p95']:>9,.0f}ms
{'first visible text — streaming':<34}{STREAM['ttft_p50']:>9,.0f}ms{STREAM['ttft_p95']:>9,.0f}ms

-> {STREAM['total_p50']/max(STREAM['ttft_p50'],1):.1f}x sooner to first visible text at the median.

Nothing got faster. The reader just stopped waiting in silence.
Worth a great deal on a screen a human is watching. Worth nothing at all
on a path where nobody is.
""")

## 7 · Output length is the lever you actually control

Generation time is dominated by how many tokens come out. This measures it
directly, at three lengths.

The slope is the number to remember, because it is what makes the next cell —
chain of thought — a latency decision rather than a quality one.

In [ ]:
LENGTHS = [80, 240, 600]
by_len = {}
print(f"{'max_tokens':>11} {'p50 ms':>9} {'out tok':>9} {'ms / token':>12}")
print("-" * 46)
for L in LENGTHS:
    got, toks = [], []
    for _ in range(N_LENGTH):
        ms, _, u = one_call(max_tokens=L,
                            task=f"{TASK} Write about {L//4} words.")
        if "error" in u:
            continue
        got.append(ms); toks.append(u.get("completion_tokens", L//2))
    if not got:
        continue
    p50 = float(np.percentile(got, 50)); mt = float(np.mean(toks))
    by_len[L] = {"p50": p50, "tokens": mt}
    print(f"{L:>11} {p50:>9,.0f} {mt:>9,.0f} {p50/max(mt,1):>12,.1f}")

if len(by_len) >= 2:
    ks = sorted(by_len)
    lo, hi = by_len[ks[0]], by_len[ks[-1]]
    dt = hi["tokens"] - lo["tokens"]
    dms = hi["p50"] - lo["p50"]
    if dt < 15:
        print(f"""
The model returned about the same number of tokens at every setting
({lo['tokens']:,.0f} vs {hi['tokens']:,.0f}), so max_tokens was not the binding
constraint — the task was. That is itself worth knowing: raising max_tokens
buys you nothing until the model actually wants to write more.

Change the task text to ask for more, and run this again.
""")
    else:
        print(f"""
Between {ks[0]} and {ks[-1]} max_tokens the output grew by {dt:,.0f} tokens
and the latency grew by {dms:,.0f} ms — about {dms/max(dt,1):,.1f} ms per token.

Every extra token you ask for is latency you have chosen to spend.
""")

## 8 · What chain of thought actually costs

Chain of thought is **more output tokens from the same call** — not a parallel
step. Modelling it as a parallel stage makes it look free, which is the mistake
this cell exists to prevent.

Same case, same model, two prompts.

In [ ]:
PLAIN = "Summarise the claim in three sentences for a handler. Cite page numbers."
COT   = ("Think step by step. First list every figure and its page. Then check "
         "each against the policy limits. Then note any inconsistency. Then "
         "write a three sentence summary for a handler, citing page numbers.")

def bench(task, n=4, max_tokens=500):
    got, toks = [], []
    for _ in range(n):
        ms, txt, u = one_call(max_tokens=max_tokens, task=task)
        if "error" in u:
            continue
        got.append(ms); toks.append(u.get("completion_tokens", 0))
    return (float(np.percentile(got,50)) if got else 0.0,
            float(np.mean(toks)) if toks else 0.0)

p_ms, p_tok = bench(PLAIN)
c_ms, c_tok = bench(COT)

print(f"{'':<26}{'p50 ms':>10}{'out tokens':>13}")
print("-" * 50)
print(f"{'plain summary':<26}{p_ms:>10,.0f}{p_tok:>13,.0f}")
print(f"{'chain of thought':<26}{c_ms:>10,.0f}{c_tok:>13,.0f}")
print("-" * 50)
if p_ms:
    print(f"{'difference':<26}{c_ms-p_ms:>+10,.0f}{c_tok-p_tok:>+13,.0f}")
    print(f"\nChain of thought cost {c_ms-p_ms:,.0f} ms at the median "
          f"({c_ms/max(p_ms,1):.2f}x).")
print("""
That is a trade with a number on both sides. Whether the extra reasoning is
worth the seconds is a decision somebody should make deliberately, and record.
""")

COT_MULTIPLIER = (c_ms / p_ms) if p_ms else 1.7
if COT_MULTIPLIER < 1.0:
    print(f"\nMeasured multiplier came out below 1.0 ({COT_MULTIPLIER:.2f}). With only "
          f"{4} samples each that is noise, not a finding — chain of thought cannot "
          f"be faster than the same call with fewer tokens.\nRaise the sample count, "
          f"or treat this number as unmeasured.")
    COT_MULTIPLIER = 1.0

## 9 · Compose the pipeline

Now put your measured generation alongside the other stages.

**Be clear about what is what.** Generation is measured against your endpoint.
The other stages are modelled from the Day 16 lab unless you supply real
timings — edit `OTHER_STAGES` if you have them.

In [ ]:
# median_ms, sigma, tail probability, tail multiplier, dependencies
OTHER_STAGES = {
    "blob_upload":      (400,  0.35, 0.03, 6.0, []),
    "doc_intelligence": (9000, 0.45, 0.06, 4.5, ["blob_upload"]),
    "redact_pii":       (250,  0.30, 0.02, 3.0, ["doc_intelligence"]),
    "search_retrieve":  (320,  0.40, 0.05, 5.0, ["blob_upload"]),
    "prompt_assemble":  (60,   0.25, 0.01, 2.5, ["redact_pii", "search_retrieve"]),
    "validate_schema":  (45,   0.20, 0.01, 2.0, ["generation"]),
    "audit_write":      (180,  0.30, 0.03, 4.0, ["validate_schema"]),
    "persist_profile":  (140,  0.30, 0.02, 3.5, ["audit_write"]),
}
GENERATION_DEPS = ["prompt_assemble"]

RUNS = 20000
rng = np.random.default_rng(11)

def sample_stage(spec, n):
    med, sig, tp, tm, _ = spec
    v = np.exp(np.log(med) + rng.normal(0, sig, n))
    hit = rng.random(n) < tp
    v[hit] *= tm
    return v

# Generation: resample from YOUR measured calls (bootstrap), so the real
# distribution shape — tail and all — flows into the composition.
def sample_generation(n, multiplier=1.0):
    return rng.choice(gen, size=n, replace=True) * multiplier

def run_pipeline(parallel=True, cot=False, overrides=None, tail_p=None,
                 tail_mult=None, seed=11):
    """
    Reset the generator each time so two configurations are compared on the
    SAME random draws. Without this you are comparing two different days and
    a 2% difference could be either the change or the weather.
    """
    global rng
    rng = np.random.default_rng(seed)
    overrides = overrides or {}; tail_p = tail_p or {}; tail_mult = tail_mult or {}
    per = {}
    for name, spec in OTHER_STAGES.items():
        med, sig, tp, tm, deps = spec
        spec2 = (med * overrides.get(name, 1.0), sig,
                 tp * tail_p.get(name, 1.0),
                 1.0 + (tm - 1.0) * tail_mult.get(name, 1.0), deps)
        per[name] = sample_stage(spec2, RUNS)
    per["generation"] = sample_generation(
        RUNS, overrides.get("generation", 1.0) * (COT_MULTIPLIER if cot else 1.0))

    deps = {n: s[4] for n, s in OTHER_STAGES.items()}
    deps["generation"] = GENERATION_DEPS
    order = ["blob_upload", "doc_intelligence", "redact_pii", "search_retrieve",
             "prompt_assemble", "generation", "validate_schema", "audit_write",
             "persist_profile"]

    finish = {}
    for name in order:
        if parallel:
            start = np.max(np.vstack([finish[d] for d in deps[name]]), axis=0) \
                    if deps[name] else np.zeros(RUNS)
        else:
            start = np.max(np.vstack(list(finish.values())), axis=0) \
                    if finish else np.zeros(RUNS)
        finish[name] = start + per[name]
    total = np.max(np.vstack(list(finish.values())), axis=0)
    return total, per

def pct(a, p): return float(np.percentile(a, p))

total_seq, per_stage = run_pipeline(parallel=False)
total_par, _ = run_pipeline(parallel=True)

print(f"{'':<22}{'P50':>11}{'P95':>11}{'P99':>11}   vs a 5s target")
print("-" * 72)
for lbl, arr in (("ingestion sequential", total_seq), ("ingestion parallel", total_par)):
    p95 = pct(arr, 95)
    verdict = "PASS" if p95 < 5000 else f"FAIL — {p95/5000:.1f}x over"
    print(f"{lbl:<22}{pct(arr,50)/1000:>10,.1f}s{p95/1000:>10,.1f}s"
          f"{pct(arr,99)/1000:>10,.1f}s   {verdict}")

# The interactive path: read a stored profile and render it. No model call.
INTERACTIVE = {"authn": (45,0.30,0.02,3.0), "read_profile": (110,0.35,0.03,5.0),
               "read_citations": (90,0.35,0.03,4.0), "render": (55,0.25,0.01,2.5)}
ia = np.zeros(RUNS)
for spec in INTERACTIVE.values():
    ia += sample_stage((*spec, []), RUNS)
p95i = pct(ia, 95)
print(f"{'interactive':<22}{pct(ia,50):>10,.0f}ms{p95i:>10,.0f}ms{pct(ia,99):>10,.0f}ms"
      f"   {'PASS' if p95i < 5000 else 'FAIL'}")

## 10 · The finding your brief was pointing at

Two paths, and only one of them has a person waiting on it.

```
INGESTION      PDF arrives -> extract -> retrieve -> generate -> store
               Nobody is watching. Tens of seconds. Asynchronous.

INTERACTIVE    Handler opens the case -> read the stored profile -> render
               A person IS waiting. Sub-second, because the expensive work
               already happened.
```

The five-second target is not achievable on the ingestion path and is
comfortable on the interactive one. **That is not a failure to optimise. The
target is being applied to the wrong path.**

Sprint 0 Day 5 drew this line already — async for extraction, sync for the
handler screen. This is where it becomes a number you can defend.

> A latency target without a named path is a number somebody will quietly
> redefine. A team measured against the wrong path will either report a figure
> they cannot hit, or change what they are timing until they can.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.hist(total_par/1000, bins=60, color="#8955FD", alpha=0.8,
        edgecolor="white", label="ingestion path")
ax.axvline(5, color="#C0223B", linewidth=2.2, label="5 second target")
ax.axvline(pct(total_par,95)/1000, color="#FD600E", linestyle="--",
           linewidth=1.8, label=f"ingestion P95 = {pct(total_par,95)/1000:,.1f}s")
ax.set_xlabel("seconds"); ax.set_title(
    "The ingestion path against a 5 second target", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"""
ingestion   P95 {pct(total_par,95)/1000:>7,.1f}s   {pct(total_par,95)/5000:>4.1f}x the target
interactive P95 {pct(ia,95)/1000:>7,.2f}s   {pct(ia,95)/5000:>4.1f}x the target

Your NFR should read something like:

    "P95 under 5 seconds on the handler read path, measured over a rolling
     hour. The ingestion path is asynchronous and budgeted per document at
     P95 under {math.ceil(pct(total_par,95)/1000/5)*5} seconds."

Two numbers, two paths, both defensible. One number is an argument waiting
to happen.
""")

## 11 · Why P95 is not the sum of the stage P95s

A project manager will ask for a stage-by-stage latency budget. The obvious way
to build one is wrong, and it is wrong in **both** directions.

In [ ]:
sum_p50 = sum(pct(v, 50) for v in per_stage.values())
sum_p95 = sum(pct(v, 95) for v in per_stage.values())
real_p50, real_p95 = pct(total_seq, 50), pct(total_seq, 95)

print(f"{'stage':<20}{'P50':>10}{'P95':>10}")
print("-" * 42)
for name, v in per_stage.items():
    print(f"{name:<20}{pct(v,50):>9,.0f}ms{pct(v,95):>9,.0f}ms")
print("-" * 42)
print(f"{'SUM of stages':<20}{sum_p50:>9,.0f}ms{sum_p95:>9,.0f}ms")
print(f"{'ACTUAL pipeline':<20}{real_p50:>9,.0f}ms{real_p95:>9,.0f}ms")

print(f"""
  Sum of stage P95s   {sum_p95/1000:>6,.1f}s   OVERESTIMATES by {(sum_p95-real_p95)/1000:,.1f}s
  Actual pipeline P95 {real_p95/1000:>6,.1f}s
  Sum of stage P50s   {sum_p50/1000:>6,.1f}s   UNDERESTIMATES by {(real_p95-sum_p50)/1000:,.1f}s

Every stage does not have a bad day at the same time. Adding up P95s assumes
they do, and gives you a budget you can never spend. Adding up P50s assumes
they never do, and gives you a promise you cannot keep.

Neither is a rounding error. Measure the flow end to end, and treat stage
percentiles as diagnostics rather than as a contract.
""")

## 12 · P50 and P95 are moved by different work

Three changes, each aimed somewhere different.

**A** makes the typical case faster — algorithm work: a smaller payload, a
nearer region, a cheaper model.

**B and C** do not touch the typical case at all. They make slow events
**rarer** and **less severe** — operational work: warm pools, retry budgets,
timeouts, capacity headroom.

The `tail focus` column is ΔP95 ÷ ΔP50: how much P95 improvement you bought per
unit of P50 improvement.

In [ ]:
base_p50, base_p95 = pct(total_par, 50), pct(total_par, 95)

EXPERIMENTS = [
    ("baseline (parallel)",                    dict()),
    ("A. Doc Intel 30% faster (median only)",  dict(overrides={"doc_intelligence": 0.70})),
    ("B. Doc Intel slow events 5x rarer",      dict(tail_p={"doc_intelligence": 0.20})),
    ("C. Doc Intel slow events half as bad",   dict(tail_mult={"doc_intelligence": 0.50})),
    ("B + C together (attack the tail)",       dict(tail_p={"doc_intelligence": 0.20},
                                                   tail_mult={"doc_intelligence": 0.50})),
    ("add chain of thought",                   dict(cot=True)),
    ("cache retrieval (off critical path)",    dict(overrides={"search_retrieve": 0.15})),
]

print(f"{'change':<40}{'P50':>9}{'P95':>9}{'dP50':>9}{'dP95':>9}{'tail focus':>12}")
print("-" * 88)
for label, kw in EXPERIMENTS:
    t, _ = run_pipeline(parallel=True, **kw)
    p50, p95 = pct(t,50), pct(t,95)
    d50, d95 = p50-base_p50, p95-base_p95
    ratio = f"{d95/d50:.1f}x" if abs(d50) > 50 else "-"
    print(f"{label:<40}{p50/1000:>8,.1f}s{p95/1000:>8,.1f}s"
          f"{d50/1000:>+8,.1f}s{d95/1000:>+8,.1f}s{ratio:>12}")

print("""
Compare A with B and C.

A makes the typical case 30% faster. It moves P50 well and P95 less.
B and C never touch the typical case, and they move P95 far harder per unit
of effort spent.

    Optimising the median is algorithm work.
    Optimising the tail is operational work.

A team asked for P95 that goes and optimises the median will spend a sprint
and report no improvement.

And note the last row: caching retrieval changes nothing, because in parallel
mode it is not on the critical path. Optimising something off the critical
path is effort with no result attached.
""")

## 13 · What to emit, and what to alert on

One rule above all others: **never alert on the mean.**

At 4,000 documents a month, one case in twenty taking ninety seconds is two
hundred unhappy cases — and a few hundred milliseconds on the mean. A
mean-based alert sits quietly through that and fires during a harmless spike.

In [ ]:
target_p95 = 5000
alert_band = pct(ia, 95) * 1.5

print(f"""
metric                     emit from                 alert when
{'-'*84}
pipeline_duration_ms       end to end, tagged by     P95 on the INTERACTIVE path
                           path                      exceeds {target_p95:,} ms

stage_duration_ms          every stage, tagged by    a single stage's P95 leaves
                           name                      its agreed band

tail_event_count           retries, cold starts,     the RATE rises — this is the
                           oversized documents       leading indicator

schema_valid_rate          the validator             it falls at all

retrieval_empty_rate       search, per query         it rises — the failure that
                                                     produces confident wrong output

Your measured numbers, for the alert thresholds:

  generation P95            {GEN['p95']:>8,.0f} ms
  generation P99            {GEN['p99']:>8,.0f} ms
  interactive path P95      {pct(ia,95):>8,.0f} ms      alert above ~{alert_band:,.0f} ms
  ingestion path P95        {pct(total_par,95)/1000:>8,.1f} s

The last two rows of that table are not latency metrics. They are there because
a pipeline that gets faster while getting less correct looks like a success on
any dashboard that only measures time.
""")

## 14 · Export

In [ ]:
report = {
  "mode": "live" if LIVE else "simulated",
  "deployment": AOAI_DEPLOYMENT if LIVE else None,
  "generation_measured_ms": {k: round(v, 1) if isinstance(v, float) else v
                             for k, v in GEN.items()},
  "streaming_ms": {k: round(v, 1) for k, v in STREAM.items()},
  "chain_of_thought_multiplier": round(COT_MULTIPLIER, 3),
  "output_length_ms": {str(k): {kk: round(vv,1) for kk, vv in v.items()}
                       for k, v in by_len.items()},
  "pipeline_ms": {
      "ingestion_sequential": {"p50": round(pct(total_seq,50)), "p95": round(pct(total_seq,95))},
      "ingestion_parallel":   {"p50": round(pct(total_par,50)), "p95": round(pct(total_par,95))},
      "interactive":          {"p50": round(pct(ia,50)),        "p95": round(pct(ia,95))},
  },
  "composition_check": {
      "sum_of_stage_p95": round(sum_p95), "actual_pipeline_p95": round(real_p95),
      "sum_of_stage_p50": round(sum_p50),
  },
  "note": ("Generation measured against a live endpoint. Other stages modelled "
           "from the Day 16 lab. Replace OTHER_STAGES with real timings when "
           "you have them."),
}
Path("latency_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

try:
    from google.colab import files
    files.download("latency_report.json")
except Exception:
    print("\n(not in Colab — file is in the working directory)")

## 15 · Questions for the ADR

1. **Which path does your NFR apply to?** Does the wording make that
   unambiguous to somebody who was not in the room?

2. **Your P95 is not the sum of your stage P95s.** Can you explain why to a
   project manager who has just asked you for a stage-by-stage budget?

3. **Which lever moved P95, and which only moved P50?** Where are you planning
   to spend your optimisation effort, and does that match?

4. **Chain of thought cost you a measured number of milliseconds.** What does
   it buy, and who decided that trade?

5. **You measured against one deployment on one day.** What is your review
   trigger for re-measuring — a model version change, a region change, a volume
   change?

6. **What alerts on the tail?** If the answer is "we watch the average", you
   have an alert that cannot fire when it matters.

---

### One honest limitation

You measured your endpoint from a Colab VM, over the public internet, at one
moment, with one prompt shape, at low concurrency.

Production will differ on every one of those. The **shapes** transfer — tail
dominance, non-additive percentiles, output length driving generation time.
The **absolute numbers** do not.

Re-measure from inside your own network, at your own concurrency, with your own
prompts, before you put a figure in a contract.